# Week 3 — Data Engineering for AI: V1 Data Pipeline

**Project:** FYP Assistant — a document-grounded assistant for FYP administration questions
**Topic 2 requirements served:** FR1, FR2, FR3, FR5 / NFR2, NFR3 / RAI1
**Selected pathway:** **A — PDF / text** (single modality)
**Source:** `FYP Handbook.pdf` (119 pp., UTM FAI, 2026–2027)

## Why Pathway A only

The application answers questions about FYP phases, procedures, report structure and
forms, and every answer must cite a handbook section (FR3/RAI1). The only project data
is therefore document text. Pathways B–F (tabular, image, JSON, audio, time-series)
are **not** justified by any Topic 2 requirement and are deliberately not run.

## V1 ingestion scope

| Subset | Pages | Chapter | Serves |
|---|---|---|---|
| S01 | 7–28 | Ch. 1 Introduction — phases, registration, supervisor, logbook | FR1, FR5 |
| S02 | 30–44 | Ch. 3 Report writing and format — structure, page limits, Harvard | FR2 |

Pages 1–6 (cover, editorial, table of contents) and 29 (Ch. 2 project categories) carry
no rule content for these requirements. Chapters 4–9 and the appendices are **not yet
ingested** — recorded as a known limitation, not an oversight (see
`docs/week3_pipeline_decisions.md`).

## Execution order

`Cell 1 → Cell 3 → Part A inspection → Cell 4 → Cell 4A → Cell 5A1 → Cell 5A2 → B3 → B4.1 → Cell 6A → Cell 6B`
→ `Cell 7 → Cell 8A → Cell 9 → Step C1`

Before running: upload `FYP_Handbook.pdf` into `data/raw/` using the Colab **Files** panel.

## Cell 1 — Workspace setup (REQUIRED, run first)

Defines the canonical paths every later cell depends on. Run again after any runtime reset.

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


## Cell 3 — Verify uploaded inputs (REQUIRED)

A successful upload click is not evidence. This lists every file actually present under
`data/raw`, recursively, with its byte size. If `Input count: 0`, the files landed in
`/content` instead of `/content/week3_project/data/raw`.

In [2]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
 	print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 1
FYP_Handbook.pdf bytes: 89523826
Upload verified. Select your relevant input(s) for Step A3.


## Part A — A3: Inspect the selected data source (REQUIRED)

Two representative subsets of the single source are inspected. The goal is to prove the
structure and content are actually usable, and to surface ingestion limitations **before**
choosing a preparation method — not to show a "file loaded" message.

Three properties are measured because each one changes the Part B design:

1. **Empty pages** — a page with zero extractable characters means the content is an
   image. Those pages cannot be chunked and must be flagged, not silently skipped.
2. **Duplicated text layers** — this PDF was exported from Canva, which duplicates each
   text element. Both intra-line repeats (`A A A A`) and adjacent identical lines must be
   collapsed, otherwise every chunk is inflated and retrieval wastes slots on copies.
3. **Character volume per subset** — decides chunk size and tells us how many retrieval
   units the V1 knowledge base will contain.

In [3]:
# Part A — A3: Inspect the selected source (REQUIRED)
# Evidence recorded in docs/week3_data_inventory.md
import re
from pypdf import PdfReader

SOURCE_NAME = "FYP_Handbook.pdf"
pdf_path = RAW_DIR / SOURCE_NAME
assert pdf_path.is_file(), f"Upload {SOURCE_NAME} into {RAW_DIR} first."

reader = PdfReader(str(pdf_path))
meta = reader.metadata or {}

# --- Document-level properties -------------------------------------------
print("SOURCE:", SOURCE_NAME)
print("Pages:", len(reader.pages))
print("Size:", pdf_path.stat().st_size, "bytes")
print("Title:", meta.get("/Title"))
print("Producer:", meta.get("/Producer"))
print("Created:", meta.get("/CreationDate"))

raw_pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
total_chars = sum(len(t) for _, t in raw_pages)
empty_pages = [pn for pn, t in raw_pages if not t.strip()]

print("\nTotal extracted characters:", total_chars)
print("Pages with no extractable text:", empty_pages)

# --- Duplicated text layers (Canva export artefact) ----------------------
def collapse_repeats(line):
    """Collapse a line that is the same unit repeated 2-5 times."""
    n = len(line)
    if n < 6:
        return line
    for k in range(2, 6):
        if n % k == 0:
            unit = line[: n // k]
            if unit.strip() and unit * k == line:
                return unit.strip()
    return line

all_lines = [l.strip() for l in "\n".join(t for _, t in raw_pages).splitlines() if l.strip()]
adjacent_dups = sum(1 for a, b in zip(all_lines, all_lines[1:]) if a == b)
intraline_dups = sum(1 for l in all_lines if collapse_repeats(l) != l)

print("\nNon-empty lines:", len(all_lines))
print("Adjacent duplicate lines:", adjacent_dups,
      f"({100 * adjacent_dups / len(all_lines):.1f}%)")
print("Lines containing repeated text layers:", intraline_dups,
      f"({100 * intraline_dups / len(all_lines):.1f}%)")

# --- The two inspected subsets -------------------------------------------
SUBSETS = {
    "S01": (7, 28, "Chapter 1 - FYP phases, registration, supervisor, logbook"),
    "S02": (30, 44, "Chapter 3 - report structure, page limits, Harvard referencing"),
}

print("\n--- Subset inspection ---")
for sid, (a, b, purpose) in SUBSETS.items():
    pages = [(pn, t) for pn, t in raw_pages if a <= pn <= b]
    chars = sum(len(t) for _, t in pages)
    empties = [pn for pn, t in pages if not t.strip()]
    first = next((t for _, t in pages if t.strip()), "")
    sample = " ".join(first.split())[:160]
    print(f"\n{sid}: pp.{a}-{b} ({len(pages)} pages, {chars:,} chars) — {purpose}")
    print(f"    empty pages: {empties if empties else 'none'}")
    print(f"    first content: {sample!r}")

# --- Structural check: do the expected rule topics actually appear? -------
probe = {
    "logbook / meeting": r"logbook",
    "page limit": r"page limit|pages",
    "Turnitin": r"turnitin|plagiaris",
    "Harvard": r"harvard",
}
body = " ".join(" ".join(t.split()) for _, t in raw_pages if t)
print("\n--- Content coverage probe (whole source) ---")
for label, pattern in probe.items():
    print(f"    {label:22s} hits: {len(re.findall(pattern, body, re.I))}")

print("\nPart A inspection complete. Record these numbers in docs/week3_data_inventory.md")

SOURCE: FYP_Handbook.pdf
Pages: 119
Size: 89523826 bytes
Title: Handbook FYP BAI 20262027
Producer: Canva
Created: D:20261008171247+00'00'

Total extracted characters: 187529
Pages with no extractable text: [10, 16, 18, 94]

Non-empty lines: 4058
Adjacent duplicate lines: 272 (6.7%)
Lines containing repeated text layers: 57 (1.4%)

--- Subset inspection ---

S01: pp.7-28 (22 pages, 32,283 chars) — Chapter 1 - FYP phases, registration, supervisor, logbook
    empty pages: [10, 16, 18]
    first content: 'The Final Year Project (FYP) is a compulsory undergraduate project that students must complete and pass as part of the graduation requirements. It provides an o'

S02: pp.30-44 (15 pages, 25,430 chars) — Chapter 3 - report structure, page limits, Harvard referencing
    empty pages: none
    first content: 'This chapter outlines the thesis writing guideline for both courses. It serves as a main reference for students in conducting their projects. However, students '

--- Content coverag

## Part B — Cell 4: Source selection + lineage utilities (REQUIRED)

`select_source()` refuses ambiguous input, `source_hash()` fingerprints a file, and
`save_manifest()` writes the lineage record. Never take the first file matching an
extension — that silently loads the wrong dataset.

In [4]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


## Cell 4A — Initialise the common lineage registry (REQUIRED)

Run once, immediately after Cell 4. It prepares the registration mechanism; it does not
process data or write the manifest. The manifest is written in Cell 6A.

In [5]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


## Cell 5A1 — Pathway A1: parse, clean, and compare chunking (CONDITIONAL REQUIRED)

Three preparation configurations are computed on the same input so Step B3 can compare
them with real numbers:

| Config | Method |
|---|---|
| `fixed` | 500-character slices across the concatenated document |
| `structured` | paragraph-aware (blank-line) 500-character chunks — the lab's default |
| `paged` | paragraph-aware, line-boundary aware, 800-character cap, **never crossing a page** |

The third configuration is the one expected to win, because `pypdf` emits a single
newline per visual line and **no blank lines between paragraphs** — so blank-line
paragraph detection degenerates on this PDF, while page boundaries remain reliable and
are also what FR3/RAI1 need for citation.

In [6]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
f = select_source(relative_path="FYP_Handbook.pdf")

# Step 2: Restrict to the V1 ingestion scope (see notebook header)
PAGE_RANGES = [(7, 28), (30, 44)]

def in_scope(page_number):
    return any(a <= page_number <= b for a, b in PAGE_RANGES)

reader = PdfReader(str(f))
pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
scoped = [(pn, t) for pn, t in pages if in_scope(pn)]

print("Source:", f.name)
print("Pages in source:", len(pages))
print("Pages in V1 scope:", len(scoped))
print("Scoped pages with no text:", [pn for pn, t in scoped if not t.strip()])

# Step 3: Basic text cleaning
# 3a. The PDF exporter duplicates every text element. Collapse a line that is
#     the same unit repeated several times ("A A A A" -> "A").
def collapse_repeats(line):
    n = len(line)
    if n < 6:
        return line
    for k in range(2, 6):
        if n % k == 0:
            unit = line[: n // k]
            if unit.strip() and unit * k == line:
                return unit.strip()
    return line

# 3b. Drop adjacent duplicate lines and normalise internal whitespace.
def clean_page(text):
    out = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", collapse_repeats(line.strip()))
        if not line:
            continue
        if out and out[-1] == line:
            continue
        out.append(line)
    return "\n".join(out)

cleaned = [(pn, clean_page(t)) for pn, t in scoped]

raw_chars = sum(len(t) for _, t in scoped)
clean_chars = sum(len(t) for _, t in cleaned)

print("Raw characters:", raw_chars)
print("Cleaned characters:", clean_chars)
print("Reduction: %.1f%%" % (100 * (1 - clean_chars / raw_chars)))

assert clean_chars > 0, "No usable text: check extraction or OCR needs."

# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]

# Step 5: Paragraph-aware chunking (blank-line based)
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

# Step 6: Paragraph-aware chunking that respects line and page boundaries
def page_chunks(text, size=800):
    """Accumulate whole lines up to `size`; never split a line unless it is
    longer than `size` on its own."""
    parts = []
    current = ""
    for line in text.splitlines():
        if not line.strip():
            continue
        if len(current) + len(line) + 1 <= size:
            current += ("" if not current else "\n") + line
        else:
            if current:
                parts.append(current)
            if len(line) > size:
                parts.extend(fixed_chunks(line, size))
                current = ""
            else:
                current = line
    if current:
        parts.append(current)
    return parts

# Step 7: Compare the configurations
document_text = "\n\n".join(t for _, t in cleaned)

fixed = fixed_chunks(document_text, size=500)
structured = paragraph_chunks(document_text, size=500)
paged = [(pn, c) for pn, t in cleaned for c in page_chunks(t, 800) if c.strip()]

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))
print("Page-aware chunks:", len(paged))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPage-aware sample (p%d):" % paged[0][0])
print(repr(paged[0][1][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: FYP_Handbook.pdf
Pages in source: 119
Pages in V1 scope: 37
Scoped pages with no text: [10, 16, 18]
Raw characters: 57713
Cleaned characters: 56595
Reduction: 1.9%

--- Chunking Comparison ---
Fixed chunks: 114
Paragraph-aware chunks: 132
Page-aware chunks: 90

Fixed sample:
'The Final Year Project (FYP) is a compulsory undergraduate project that students\nmust complete and pass as part of the graduation requirements. It provides an\nopportunity to integr'

Paragraph-aware sample:
'The Final Year Project (FYP) is a compulsory undergraduate project that students\nmust complete and pass as part of the graduation requirements. It provides an\nopportunity to integr'

Page-aware sample (p7):
'The Final Year Project (FYP) is a compulsory undergraduate project that students\nmust complete and pass as part of the graduation requirements. It provides an\nopportunity to integr'

PATHWAY A1 completed successfully.


## Cell 5A2 — Pathway A2: export traceable chunks (REQUIRED after A1)

The chosen configuration is written to `prepared_text.jsonl`. Every record carries the
source filename, the source SHA-256, the processing label **and the page number**, so a
later answer can cite `FYP Handbook, p. 14` — which is exactly what FR3/RAI1 demand.

This produces preparation artefacts only — no embeddings and no vector index.

In [7]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json

# Step 1: Select the preparation approach (justified in Step B3)
CHOSEN_METHOD = "paragraph-aware-page"

if CHOSEN_METHOD == "paragraph-aware-page":
    chosen = paged
    approach = "paragraph-aware-char-800-page-bound"
elif CHOSEN_METHOD == "paragraph-aware":
    chosen = [(None, c) for c in structured]
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = [(None, c) for c in fixed]
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware-page', 'paragraph-aware' or 'fixed'")

# Step 2: Create traceable records
source_sha = source_hash(f)
records = [
    {
        "record_id": f"{f.stem}-p{page:03d}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_sha,
        "processing": approach,
        "page": page
    }
    for i, (page, chunk) in enumerate(chosen, 1)
]

# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("Distinct pages covered:", len({r["page"] for r in records}))
print("First record:")
for key, value in records[0].items():
    shown = (value[:140] + "...") if key == "text" else value
    print("   ", key, "=", shown)

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-800-page-bound
Total chunks exported: 90
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 78895 bytes
Validation: PASS
Distinct pages covered: 34
First record:
    record_id = FYP_Handbook-p007-0001
    text = The Final Year Project (FYP) is a compulsory undergraduate project that students
must complete and pass as part of the graduation requiremen...
    source = FYP_Handbook.pdf
    source_sha256 = 38ad807bc75cb9b4fde41aa3454b0b653e4fd4703d0b0ed66595a4dd6bbe30ca
    processing = paragraph-aware-char-800-page-bound
    page = 7


## Step B3 — Basic output contract (REQUIRED)

Code running without an exception does not prove valid output. This checks the file
exists and is non-empty; the content is verified by inspection in the next cell.

In [8]:
# Step B3 — Basic output check (REQUIRED for all pathways)
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 78895
Validation: file exists and is non-empty; verify content manually too.


## Step B3 — Compare two preparation configurations (REQUIRED)

Same input, three configurations. The measured differences decide the choice: a
configuration that cuts most chunks mid-sentence and carries no page provenance cannot
support a citation requirement, however many chunks it produces.

In [9]:
# Step B3 — Compare preparation configurations on the same input
import re

def ends_mid_sentence(chunk):
    return not re.search(r"[.!?:;)]\s*$", chunk.strip())

def describe(label, chunks, has_page):
    total = len(chunks)
    lens = [len(c) for _, c in chunks] if has_page else [len(c) for c in chunks]
    texts = [c for _, c in chunks] if has_page else list(chunks)
    cut = sum(ends_mid_sentence(t) for t in texts[:-1])
    print(f"{label}")
    print(f"    chunks                : {total}")
    print(f"    mean length (chars)   : {sum(lens) / total:.0f}")
    print(f"    min / max length      : {min(lens)} / {max(lens)}")
    print(f"    cut mid-sentence      : {cut} ({100 * cut / max(total - 1, 1):.0f}%)")
    print(f"    records with page no. : {'100%' if has_page else '0%'}")
    return {
        "config": label,
        "chunks": total,
        "mean_len": round(sum(lens) / total),
        "cut_pct": round(100 * cut / max(total - 1, 1)),
        "page_provenance": has_page,
    }

print("--- Observed evidence ---\n")
rows = [
    describe("A: fixed-500 (whole document)", fixed, False),
    describe("B: paragraph-aware-500 (blank-line)", structured, False),
    describe("C: paragraph-aware-800 (page-bound)", paged, True),
]

print("\n--- Decision ---")
print("Selected: C — paragraph-aware, page-bound, 800-char cap")
print("Benefit : every record carries a page number, so an answer can cite a")
print("          handbook page (FR3/RAI1), and chunks never straddle two pages.")
print("Accepted: more records than A, so slightly higher retrieval and storage cost;")
print("          a page is the smallest citable unit, so section-level citation is")
print("          still approximate until heading detection is added.")

--- Observed evidence ---

A: fixed-500 (whole document)
    chunks                : 114
    mean length (chars)   : 497
    min / max length      : 167 / 500
    cut mid-sentence      : 110 (97%)
    records with page no. : 0%
B: paragraph-aware-500 (blank-line)
    chunks                : 132
    mean length (chars)   : 429
    min / max length      : 13 / 500
    cut mid-sentence      : 122 (93%)
    records with page no. : 0%
C: paragraph-aware-800 (page-bound)
    chunks                : 90
    mean length (chars)   : 628
    min / max length      : 31 / 800
    cut mid-sentence      : 60 (67%)
    records with page no. : 100%

--- Decision ---
Selected: C — paragraph-aware, page-bound, 800-char cap
Benefit : every record carries a page number, so an answer can cite a
          handbook page (FR3/RAI1), and chunks never straddle two pages.
Accepted: more records than A, so slightly higher retrieval and storage cost;
          a page is the smallest citable unit, so section-level c

## Step B4.1 — Register the selected pathway's lineage (REQUIRED)

Records source → prepared output with the exact processing description, so the manifest
can be regenerated and audited later.

In [10]:
# B4.1 — Register Pathway A (PDF/TXT)
register_lineage(
    source=f,
    output=prepared_path,
    processing=(
        f"{approach}; "
        "pages=7-28,30-44; "
        "collapse_duplicate_text_layers=True; "
        "drop_adjacent_duplicate_lines=True; "
        "whitespace_normalised=True"
    )
)

Registered: FYP_Handbook.pdf -> prepared_text.jsonl


## Cell 6A — Generate the common lineage manifest (REQUIRED)

Combines every registered pathway into one `week3_manifest.json` with source and output
SHA-256 fingerprints.

In [11]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/week3_project/data/week3_manifest.json


## Cell 6B — Verify the saved manifest (REQUIRED)

Confirms the manifest exists, parses, and that every recorded checksum still matches the
files on disk. This proves the lineage is intact, **not** that the data is high quality or
legally redistributable.

In [12]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Build the executable V1 pipeline

Part B produced a *prepared artefact*. Part C turns the same steps into a **registered,
re-runnable pipeline**: common utilities in Cell 7, the selected document pathway in
Cell 8A, and a repeatability check in Cell 9 that re-executes the pipeline and compares
source fingerprints, output fingerprints and source-to-output mappings.

Only Cell 8A is run. Cell 8B–8F are conditional on data types this project does not have.

## Cell 7 — Common V1 pipeline utilities (REQUIRED)

Defines the paths, SHA-256 helper, explicit source resolver, and the generic
`execute_pipeline()` / `register_and_run()` runner that enforces the stage contracts
(ingest → inspect → prepare → validate → lineage) for every pathway. No dataset is
processed at this stage.

In [13]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


def portable_path(path):
    """Record project-relative paths so the manifest is portable and identical
    whether the pipeline runs in Colab or on a local machine."""
    try:
        rel = Path(path).relative_to(BASE_DIR)
        return str(rel).replace("\\", "/")
    except ValueError:
        return str(path).replace("\\", "/")


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": portable_path(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            portable_path(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


## Cell 8A — PDF / TXT pathway (CONDITIONAL REQUIRED)

Two things are carried over from Part B verbatim, because the lab requires the Part C
pipeline to reproduce the Part B decision rather than re-decide it:

- **The same cleaning rules** (C1 collapse repeated text layers, C2 drop adjacent duplicate
  lines, C3 whitespace normalisation).
- **The same chunking configuration**: paragraph-aware, 800-character cap, page-constrained.

The lab's template supports only `fixed` and `paragraph-aware` and sets `page: None`. This
implementation extends the `paragraph-aware` branch with a page constraint, because Part B
measured that the unconstrained version degenerates on this PDF (132 fragments, minimum 13
characters) and because page provenance is required by FR3/RAI1. The configuration is
declared in `CONFIG_A` and recorded in the manifest, so the difference is auditable rather
than hidden.

In [14]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Matches the configuration selected in Part B (Step B3).
SOURCE_A = resolve_source("FYP_Handbook.pdf")

CONFIG_A = {
    "chunk_size": 800,
    "method": "paragraph-aware-page",
    "page_ranges": [[7, 28], [30, 44]],
}


# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]

    if src.suffix.lower() != ".pdf":
        raise ValueError("Expected a PDF source for Pathway A")

    reader = PdfReader(str(src))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
    in_scope = [
        (pn, t) for pn, t in pages
        if any(a <= pn <= b for a, b in cfg["page_ranges"])
    ]
    text = "\n".join(t for _, t in in_scope)

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "pages_total": len(pages),
        "pages_in_scope": len(in_scope),
        "pages_without_text": [
            pn for pn, t in in_scope if not t.strip()
        ],
        "characters_in_scope": len(text),
    }


# Step 3: Define document preparation (identical to Part B)
def collapse_repeats(line):
    """Collapse a line that is the same unit repeated 2-5 times."""
    n = len(line)
    if n < 6:
        return line
    for k in range(2, 6):
        if n % k == 0:
            unit = line[: n // k]
            if unit.strip() and unit * k == line:
                return unit.strip()
    return line


def clean_page(text):
    out = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", collapse_repeats(line.strip()))
        if not line:
            continue
        if out and out[-1] == line:
            continue
        out.append(line)
    return "\n".join(out)


def page_chunks(text, size):
    parts = []
    current = ""
    for line in text.splitlines():
        if not line.strip():
            continue
        if len(current) + len(line) + 1 <= size:
            current += ("" if not current else "\n") + line
        else:
            if current:
                parts.append(current)
            if len(line) > size:
                parts.extend(
                    line[i:i + size]
                    for i in range(0, len(line), size)
                )
                current = ""
            else:
                current = line
    if current:
        parts.append(current)
    return parts


def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    size = int(cfg["chunk_size"])

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    if cfg["method"] != "paragraph-aware-page":
        raise ValueError(
            "This project selected 'paragraph-aware-page' in Part B"
        )

    reader = PdfReader(str(src))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
    in_scope = [
        (pn, t) for pn, t in pages
        if any(a <= pn <= b for a, b in cfg["page_ranges"])
    ]

    chunks = []
    for pn, raw in in_scope:
        for chunk in page_chunks(clean_page(raw), size):
            if chunk.strip():
                chunks.append((pn, chunk))

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, (page, chunk) in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": portable_path(src),
                "text": chunk,
                "page": page
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": portable_path(src),
            "output": portable_path(target)
        }
    ]

    processing_method = (
        f"paragraph-aware text chunks of {size} characters, "
        f"page-bound; pages={cfg['page_ranges']}"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


## Cell 9 — Repeatability and provenance verification (REQUIRED)

Re-executes every registered pathway and asserts that source fingerprints, prepared-output
fingerprints, sizes, output counts and source-to-output mappings are unchanged. Execution
timestamps are expected to differ and are not compared.

This is a repeatability smoke test — it does not prove cross-environment reproducibility or
data quality.

In [15]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


## Step C1 — Expected evidence

| Check | Evidence in this notebook |
|---|---|
| Successful execution message | `SUCCESS: documents | Inputs: 1 | Outputs: 1` from Cell 8A |
| Non-empty prepared output | `data/pipeline_v1/documents/prepared_text.jsonl` |
| `pipeline_manifest.json` per pathway | `data/pipeline_v1/documents/pipeline_manifest.json` |
| PASS repeatability messages | Cell 9 output |
| Representative record inspected | printed below |

A non-empty file does not establish correctness, so one record is opened and read.

In [16]:
# Step C1 — Inspect a representative prepared record
import json
from pathlib import Path

pipe_out = PIPE_DIR / "documents" / "prepared_text.jsonl"
assert pipe_out.is_file(), "Part C prepared output missing"

with pipe_out.open(encoding="utf-8") as fh:
    pipe_rows = [json.loads(line) for line in fh if line.strip()]

print("Records:", len(pipe_rows))
print("Pages:", min(r["page"] for r in pipe_rows), "-", max(r["page"] for r in pipe_rows))
print("All records have a page:", all(r["page"] for r in pipe_rows))
print("All records non-empty:", all(r["text"].strip() for r in pipe_rows))
print("\nRepresentative record (index 40):")
sample = pipe_rows[40]
print("  id    :", sample["id"])
print("  page  :", sample["page"])
print("  chars :", len(sample["text"]))
print("  text  :", sample["text"][:400].replace("\n", " "))

# Compare Part B and Part C outputs: same decision, two artefacts
part_b = PROCESSED_DIR / "prepared_text.jsonl"
if part_b.is_file():
    with part_b.open(encoding="utf-8") as fh:
        b_rows = [json.loads(line) for line in fh if line.strip()]
    same_text = [r["text"] for r in b_rows] == [r["text"] for r in pipe_rows]
    print("\nPart B records:", len(b_rows), "| Part C records:", len(pipe_rows))
    print("Identical chunk sequence:", same_text)
    print("Page sequence identical:",
          [r["page"] for r in b_rows] == [r["page"] for r in pipe_rows])

Records: 90
Pages: 7 - 44
All records have a page: True
All records non-empty: True

Representative record (index 40):
  id    : 41
  page  : 25
  chars : 663
  text  : This category aims to develop students' competencies in scientific inquiry, experimental design, critical thinking, data analysis, reproducibility, interpretation of results, and technical communication. The use of existing AI tools, Application Programming Interfaces (APIs), pretrained models, Large Language Models (LLMs), cloud-based AI services, or open-source frameworks is permitted where appr

Part B records: 90 | Part C records: 90
Identical chunk sequence: True
Page sequence identical: True


## Part C complete — what now exists

| Artefact | Location |
|---|---|
| Executed pipeline outputs | `data/pipeline_v1/documents/prepared_text.jsonl` |
| Pipeline run manifest (provenance + repeatability) | `data/pipeline_v1/documents/pipeline_manifest.json` |
| Pipeline diagram | `docs/week3_pipeline_diagram.png` |
| Decisions, stage contracts, trade-offs | `docs/week3_pipeline_decisions.md` |
| Data inventory and inspection evidence | `docs/week3_data_inventory.md` |

The Part B lineage manifest (`data/week3_manifest.json`) records *data preparation*; the
Part C `pipeline_manifest.json` records *pipeline execution*. They are deliberately separate.